# AI Scategorizer: experiments for the paper

Runs every experiment end to end on a free Colab GPU (~30–40 min).

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**.
2. On your PC, zip the `research` folder (it must contain `data/dataset.csv`) into **`research.zip`**.
3. Run the cells in order. The first cell asks you to upload `research.zip`.

In [ ]:
from google.colab import files
uploaded = files.upload()          # choose research.zip
!rm -rf research && unzip -q research.zip -d . && ls research research/data

In [ ]:
!pip -q install "transformers>=4.46" "accelerate>=0.30" scikit-learn scipy pandas matplotlib
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

## 1. Fixed train/val/test split (same split for every model)

In [ ]:
%cd research
!python split_dataset.py --input data/dataset.csv --out data/splits
!cat data/splits/split_info.json

## 2. Baseline: TF-IDF + Logistic Regression

In [ ]:
!python train_baseline.py --data-dir data/splits --out results/tfidf_lr

## 3. Transformers, 3 seeds each
DistilBERT (small, fast) vs RoBERTa-base (larger). Each seed takes a few minutes on a T4.

In [ ]:
import os
def check(out):
    # `!` commands never raise, so stop here if training failed instead of silently continuing
    assert os.path.exists(f"{out}/predictions_test.csv"), f"Training failed for {out}: scroll up for the error and send it"

MODELS = {"distilbert": "distilbert-base-uncased", "roberta": "roberta-base"}
SEEDS = [42, 1, 2]
for short, hf_name in MODELS.items():
    for seed in SEEDS:
        out = f"results/{short}_s{seed}"
        print(f"===== {hf_name} seed {seed} =====")
        !python train_transformer.py --model-name {hf_name} --data-dir data/splits --out {out} --seed {seed} --epochs 5 --max-length 256
        check(out)

## 4. Cross-source generalisation (the key experiment)
All documents from one source are held out as the test set and never seen in training.
This simulates a new kind of document arriving after deployment.

In [ ]:
for src in ["support_tickets", "rvl_cdip"]:
    !python split_dataset.py --input data/dataset.csv --out data/splits_xsrc_{src} --holdout-source {src}
    !python train_baseline.py --data-dir data/splits_xsrc_{src} --out results/xsrc_{src}_tfidf_lr
    for short, hf_name in MODELS.items():
        print(f"===== cross-source {src}: {hf_name} =====")
        out = f"results/xsrc_{src}_{short}"
        !python train_transformer.py --model-name {hf_name} --data-dir data/splits_xsrc_{src} --out {out} --seed 42 --epochs 5 --max-length 256
        check(out)

## 5. Evaluation (metrics, CIs, calibration, threshold sweep, McNemar)

In [ ]:
!python evaluate.py results/tfidf_lr results/distilbert_s42 results/roberta_s42 --target-accuracy 0.95 --out results/report
!python evaluate.py results/xsrc_support_tickets_tfidf_lr results/xsrc_support_tickets_distilbert results/xsrc_support_tickets_roberta --target-accuracy 0.95 --out results/report_xsrc_support_tickets
!python evaluate.py results/xsrc_rvl_cdip_tfidf_lr results/xsrc_rvl_cdip_distilbert results/xsrc_rvl_cdip_roberta --target-accuracy 0.95 --out results/report_xsrc_rvl_cdip

## 6. Mean ± std over seeds, in-distribution split (put this table in the paper)

In [ ]:
import json, glob, re
import pandas as pd
!python evaluate.py results/distilbert_s1 results/distilbert_s2 results/roberta_s1 results/roberta_s2 --target-accuracy 0.95 --out results/report_seeds > /dev/null
rows = []
for path in sorted(glob.glob("results/*/eval/metrics.json")):
    m = json.load(open(path))
    run = path.split("/")[1]
    if run.startswith("xsrc_"):
        continue  # cross-source runs are reported in their own tables
    sr = m["selective_routing"]
    rows.append({
        "model": re.sub(r"_s\d+$", "", run),
        "accuracy": m["accuracy"], "macro_f1": m["macro_f1"],
        "ece_before": m["calibration"]["ece_before"], "ece_after": m["calibration"]["ece_after"],
        "threshold": sr["threshold_chosen_on_val"],
        "auto_routed_pct": sr["test_at_threshold"]["auto_routed_pct"],
        "auto_routed_acc": sr["test_at_threshold"]["auto_routed_accuracy"],
        "aurc": sr["aurc"],
    })
df = pd.DataFrame(rows)
summary = df.groupby("model").agg(["mean", "std"]).round(4)
display(summary)
summary.to_csv("results/report/seed_summary.csv")

## 7. Figures and download

In [ ]:
from IPython.display import Image, display
for f in ["confusion_matrix.png", "reliability.png", "threshold_tradeoff.png"]:
    display(Image(f"results/roberta_s42/eval/{f}", width=450))
for r in ["report", "report_xsrc_support_tickets", "report_xsrc_rvl_cdip"]:
    print("######## " + r)
    print(open(f"results/{r}/results.md").read())

In [ ]:
# Zip results WITHOUT model weights/checkpoints (small download)
!zip -qr results.zip results -x "*/checkpoints/*" "*/model/*"
files.download("results.zip")
# Optional: download the best transformer for the app (~250-500 MB)
# !cd results/roberta_s42 && zip -qr ../../best_model.zip model && cd ../.. ; files.download("best_model.zip")